# Phase 03G: Teacher B BLIP Seed 123 Parallel Confirmation

Huấn luyện `Salesforce/blip-image-captioning-base` trên MS COCO 2014
Train+Restval, chọn checkpoint chỉ bằng Karpathy Validation và giữ Held-out
Test niêm phong.

Phiên bản: `phase3g-v1.1-fp16-overflow-fix`.

> Bản sửa 15/09/2026: GradScaler được phép skip optimizer step, giảm
> loss scale và retry trước khi kiểm tra gradient hữu hạn.

## Kaggle inputs

1. Phase 03F output.
2. Phase 01 formal output.
3. MS COCO 2014 images.
4. BLIP offline model đúng revision đã khóa.
5. Offline wheel `pycocoevalcap==1.2`.
6. Output Phase 03G trước đó nếu cần resume.

## Runtime

- GPU T4; Internet Off.
- Chạy tuần tự Cell 1–8.
- Cell 6 có thể kéo dài nhiều Kaggle session; checkpoint tự động hỗ trợ resume.
- Không mở `mscoco_test.parquet`.

## Provenance note

Upstream model card mô tả checkpoint là COCO-pretrained. Run này được gắn
nhãn external COCO-pretrained teacher; không dùng Held-out Test để tuyên bố
đánh giá độc lập.

> Parallel conditional confirmation: kết quả chỉ được chấp nhận sau khi seed 42 độc lập vượt quality gate.


## Cell 1 — Environment, Phase 03F authorization và input discovery

### Công việc

- xác minh Phase 03F marker, 18 manifest row và training contract;
- tìm Phase 01, COCO, BLIP source, metric wheel và optional resume;
- bật offline/deterministic mode; chưa đọc caption/model/Test.

### Thời gian dự kiến

2–7 phút.

### Tiến độ hiện tại

Cell 1/8 — khóa đầu vào và quyền huấn luyện.

### Đầu ra dự kiến

```text
Status                    : PASSED
Phase 03F                 : VERIFIED
Parallel training seed 123     : AUTHORIZED
Optional resume           : 0 or 1
Test Set status           : SEALED
Progress                  : 1 / 8 cells passed
```


In [ ]:
import csv
import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from collections import defaultdict
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import transformers

STARTED = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = (
    Path("/kaggle/working") / ""
    / "phase3_teacher_b_seed123"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LOG_DIR = OUTPUT_ROOT / "logs"
VALIDATION_DIR = OUTPUT_ROOT / "validation"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for path in [OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, CHECKPOINT_DIR,
             LOG_DIR, VALIDATION_DIR, MANIFEST_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 123
EXPECTED_REPO = "Salesforce/blip-image-captioning-base"
EXPECTED_REVISION = "82a37760796d32b1411fe092ab5d4e227313294b"
DEVICE = torch.device("cuda")


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with open(path, "r", encoding="utf-8") as source:
        return json.load(source)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True)
        + "\n", encoding="utf-8"
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


assert torch.cuda.is_available()
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=False)
set_seeds(SEED)

targets = {
    "PHASE3_TEACHER_B_SEED42_FULL_TRAINING_AUTHORIZED.json": [],
    "PHASE1_FORMAL_DELIVERABLES_READY.json": [],
    "MODEL_SOURCE_MANIFEST.json": [],
    "pycocoevalcap-1.2-py3-none-any.whl": [],
    "teacher_b_seed123_last.pt": [],
    "TEACHER_B_SEED123_FULL_RUN_COMPLETE.json": [],
}
train_leaves, val_leaves = [], []
for current_text, dirs, files in os.walk(INPUT_ROOT):
    current = Path(current_text)
    if current.name == "train2014" and any(
        x.startswith("COCO_train2014_") and x.endswith(".jpg")
        for x in files
    ):
        train_leaves.append(current.resolve()); dirs[:] = []; continue
    if current.name == "val2014" and any(
        x.startswith("COCO_val2014_") and x.endswith(".jpg")
        for x in files
    ):
        val_leaves.append(current.resolve()); dirs[:] = []; continue
    for name in set(files).intersection(targets):
        targets[name].append((current / name).resolve())

assert len(targets[
    "PHASE3_TEACHER_B_SEED42_FULL_TRAINING_AUTHORIZED.json"
]) == 1
AUTH_PATH = targets[
    "PHASE3_TEACHER_B_SEED42_FULL_TRAINING_AUTHORIZED.json"
][0]
PHASE3F_ROOT = AUTH_PATH.parent
AUTH = read_json(AUTH_PATH)
assert AUTH["status"] == "ready" and AUTH["seed"] == 42
assert AUTH["model"] == EXPECTED_REPO
assert AUTH["resolved_revision"] == EXPECTED_REVISION
assert AUTH["test_set_status"] == "sealed"
assert AUTH["test_set_used"] is False

manifest_path = PHASE3F_ROOT / AUTH["manifests"]["csv"]["relative_path"]
assert sha256_file(manifest_path) == AUTH["manifests"]["csv"]["sha256"]
with open(manifest_path, newline="", encoding="utf-8") as source:
    phase3f_rows = list(csv.DictReader(source))
assert len(phase3f_rows) == 18
for row in phase3f_rows:
    path = (PHASE3F_ROOT / row["relative_path"]).resolve()
    assert path.is_relative_to(PHASE3F_ROOT.resolve())
    assert path.is_file()
    assert path.stat().st_size == int(row["size_bytes"])
    assert sha256_file(path) == row["sha256"]

TRAINING_CONTRACT_PATH = (
    PHASE3F_ROOT / AUTH["training_contract"]["relative_path"]
)
assert sha256_file(TRAINING_CONTRACT_PATH) == AUTH[
    "training_contract"
]["sha256"]
CONTRACT = read_json(TRAINING_CONTRACT_PATH)
assert CONTRACT["status"] == "locked"
assert 123 in CONTRACT["confirmation_seeds_after_seed42_gate"]
assert CONTRACT["test_set_used"] is False

PARALLEL_CONFIRMATION_MODE = True
PARALLEL_PROTOCOL_PATH = (
    CONFIG_DIR / "teacher_b_seed123_parallel_confirmation_protocol.json"
)
write_json(PARALLEL_PROTOCOL_PATH, {
    "status": "locked",
    "created_at_utc": now_utc(),
    "teacher_id": "B",
    "seed": 123,
    "parallel_confirmation_mode": True,
    "requested_by_user": True,
    "acceptance_condition": (
        "Seed 42 must independently pass its Validation quality gate."
    ),
    "same_preflight_and_training_contract": True,
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
})

assert len(targets["PHASE1_FORMAL_DELIVERABLES_READY.json"]) == 1
PHASE1_READY_PATH = targets[
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
][0]
PHASE1_ROOT = PHASE1_READY_PATH.parent
assert len(train_leaves) == 1 and len(val_leaves) == 1
COCO_TRAIN_LEAF, COCO_VAL_LEAF = train_leaves[0], val_leaves[0]

model_candidates = []
for path in targets["MODEL_SOURCE_MANIFEST.json"]:
    payload = read_json(path)
    if (payload.get("repo_id") == EXPECTED_REPO and
            payload.get("resolved_commit") == EXPECTED_REVISION):
        model_candidates.append(path.parent)
assert len(model_candidates) == 1
BLIP_SOURCE_DIR = model_candidates[0]
BLIP_SOURCE_MANIFEST_PATH = BLIP_SOURCE_DIR / "MODEL_SOURCE_MANIFEST.json"
assert len(targets["pycocoevalcap-1.2-py3-none-any.whl"]) == 1
PYCOCO_WHEEL = targets["pycocoevalcap-1.2-py3-none-any.whl"][0]
RESUME_CANDIDATES = targets["teacher_b_seed123_last.pt"]
assert len(RESUME_CANDIDATES) <= 1
assert len(targets["TEACHER_B_SEED123_FULL_RUN_COMPLETE.json"]) == 0, (
    "Đã gắn một run COMPLETE; không được huấn luyện lặp."
)

write_json(AUDIT_DIR / "phase3g_cell01_inputs.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "phase3f_manifest_rows": 18, "seed": 123,
    "model": EXPECTED_REPO, "revision": EXPECTED_REVISION,
    "resume_candidates": [str(x) for x in RESUME_CANDIDATES],
    "dataset_read": False, "model_loaded": False,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72); print("PHASE 03G CELL 1 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Transformers              :", transformers.__version__)
print("Phase 03F                 : VERIFIED")
print("Parallel training seed 123     : AUTHORIZED")
print("Phase 03F manifest        : 18 / 18 VERIFIED")
print("BLIP offline source       : FOUND")
print("Metric wheel              : FOUND")
print("Optional resume           :", len(RESUME_CANDIDATES))
print("Dataset/model read        : False / False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 8 cells passed")
print("Next                      : build development indexes")


## Cell 2 — Phase 01 data, image-balanced sampler và references

### Công việc

- xác minh Train/Validation/Image manifest;
- dựng 113.287 nhóm caption theo ảnh;
- một caption/ảnh/epoch, xoay caption deterministic;
- dựng 5.000 Validation reference; không mở Test.

### Thời gian dự kiến

1–3 phút.

### Tiến độ hiện tại

Cell 2/8 — dựng chỉ mục dữ liệu và sampler.

### Đầu ra dự kiến

```text
Status                    : PASSED
Training images           : 113287
Validation images/refs    : 5000 / 25010
Sampler repeatable        : True
Test Parquet opened       : False
Progress                  : 2 / 8 cells passed
```


In [ ]:
STARTED = time.perf_counter()
import pyarrow.parquet as pq

READY = read_json(PHASE1_READY_PATH)
assert READY["status"] == "ready" and READY["test_used"] is False
SHA_PATH = next(PHASE1_ROOT.rglob("SHA256SUMS.txt"))
assert sha256_file(SHA_PATH) == READY["sha256sums_sha256"]
sums = {}
for line in SHA_PATH.read_text(encoding="utf-8").splitlines():
    if "  " in line:
        digest, relative = line.split("  ", 1)
        sums[relative.strip().replace("\\", "/")] = digest


def verified_phase1(name):
    matches = list(PHASE1_ROOT.rglob(name))
    assert len(matches) == 1
    matches_sum = [
        value for key, value in sums.items()
        if key == name or key.endswith("/" + name)
    ]
    assert len(matches_sum) == 1
    assert sha256_file(matches[0]) == matches_sum[0]
    return matches[0]


TRAIN_PARQUET = verified_phase1("mscoco_train.parquet")
VAL_PARQUET = verified_phase1("mscoco_validation.parquet")
IMAGE_PARQUET = verified_phase1("mscoco_image_manifest.parquet")
train_rows = pq.read_table(
    TRAIN_PARQUET,
    columns=["annotation_id", "image_id", "karpathy_split",
             "source_partition", "caption"],
).to_pylist()
VAL_RECORDS = pq.read_table(
    VAL_PARQUET,
    columns=["annotation_id", "image_id", "karpathy_split",
             "source_partition", "caption"],
).to_pylist()
image_rows = pq.read_table(
    IMAGE_PARQUET,
    columns=["image_id", "experiment_split", "source_partition",
             "image_path"],
).to_pylist()
assert len(train_rows) == 566747 and len(VAL_RECORDS) == 25010


def live_path(row):
    name = Path(row["image_path"]).name
    return (
        COCO_TRAIN_LEAF / name if row["source_partition"] == "train2014"
        else COCO_VAL_LEAF / name
    )


TRAIN_PATHS, VAL_PATHS, TEST_IDS = {}, {}, set()
for row in image_rows:
    image_id = int(row["image_id"])
    if row["experiment_split"] == "train":
        TRAIN_PATHS[image_id] = live_path(row)
    elif row["experiment_split"] == "validation":
        VAL_PATHS[image_id] = live_path(row)
    elif row["experiment_split"] == "held_out_test":
        TEST_IDS.add(image_id)
assert (len(TRAIN_PATHS), len(VAL_PATHS), len(TEST_IDS)) == (
    113287, 5000, 5000
)
assert not (set(TRAIN_PATHS) & set(VAL_PATHS))
assert not (set(TRAIN_PATHS) & TEST_IDS)
assert all(x.is_file() for x in list(TRAIN_PATHS.values())[:256])
assert all(x.is_file() for x in list(VAL_PATHS.values())[:256])

TRAIN_BY_IMAGE = defaultdict(list)
for row in train_rows:
    image_id = int(row["image_id"])
    assert image_id in TRAIN_PATHS and image_id not in TEST_IDS
    TRAIN_BY_IMAGE[image_id].append({
        "annotation_id": int(row["annotation_id"]),
        "image_id": image_id,
        "caption": row["caption"],
        "image_path": TRAIN_PATHS[image_id],
    })
for values in TRAIN_BY_IMAGE.values():
    values.sort(key=lambda x: x["annotation_id"])
assert len(TRAIN_BY_IMAGE) == 113287
TRAIN_IMAGE_IDS = sorted(TRAIN_BY_IMAGE)


def selected_epoch_records(epoch, cursor=0):
    image_ids = list(TRAIN_IMAGE_IDS)
    random.Random(SEED + epoch * 1000003).shuffle(image_ids)
    records = []
    for image_id in image_ids:
        options = TRAIN_BY_IMAGE[image_id]
        index = (SEED + epoch) % len(options)
        records.append(options[index])
    return records[cursor:]


probe_a = selected_epoch_records(0)
probe_b = selected_epoch_records(0)
probe_c = selected_epoch_records(1)
assert [x["image_id"] for x in probe_a] == [x["image_id"] for x in probe_b]
assert set(x["image_id"] for x in probe_a) == set(
    x["image_id"] for x in probe_c
)
assert len({x["image_id"] for x in probe_a}) == 113287

REFERENCES = defaultdict(list)
for row in VAL_RECORDS:
    image_id = int(row["image_id"])
    assert image_id in VAL_PATHS and image_id not in TEST_IDS
    REFERENCES[image_id].append({"caption": row["caption"]})
assert len(REFERENCES) == 5000
assert sum(map(len, REFERENCES.values())) == 25010

write_json(AUDIT_DIR / "phase3g_cell02_data_sampler.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "training_captions": len(train_rows),
    "training_images": len(TRAIN_BY_IMAGE),
    "validation_captions": len(VAL_RECORDS),
    "validation_images": len(REFERENCES),
    "sampler_repeatable": True, "captions_per_image_per_epoch": 1,
    "test_parquet_opened": False, "test_set_status": "sealed",
    "test_set_used": False, "elapsed_seconds": time.perf_counter() - STARTED,
})
del image_rows, probe_a, probe_b, probe_c
print("=" * 72); print("PHASE 03G CELL 2 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("Training captions/images  :", len(train_rows), "/", len(TRAIN_BY_IMAGE))
print("Validation refs/images    :", sum(map(len, REFERENCES.values())), "/", len(REFERENCES))
print("Sampler repeatable        : True")
print("Captions/image/epoch      : 1")
print("Test Parquet opened       : False")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 8 cells passed")
print("Next                      : load processor, model and dataloaders")


## Cell 3 — Processor, model và DataLoaders

### Công việc

- xác minh BLIP source checksum và load offline;
- thêm `[DEC]/[ENC]` đúng ID cho Transformers 5;
- dựng Train/Validation datasets với ảnh 384×384 và text length 40;
- smoke-test batch; không optimizer step.

### Thời gian dự kiến

2–5 phút.

### Tiến độ hiện tại

Cell 3/8 — dựng pipeline end-to-end.

### Đầu ra dự kiến

```text
Status                    : PASSED
Parameters                : 247444600
Physical/effective batch  : 2 / 32
Train/Validation batch    : PASSED
Progress                  : 3 / 8 cells passed
```


In [ ]:
STARTED = time.perf_counter()
from PIL import Image
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import BlipForConditionalGeneration, BlipProcessor

source_manifest = read_json(BLIP_SOURCE_MANIFEST_PATH)
assert source_manifest["resolved_commit"] == EXPECTED_REVISION
for item in source_manifest["files"]:
    path = BLIP_SOURCE_DIR / item["relative_path"]
    assert path.is_file() and path.stat().st_size == item["size_bytes"]
    assert sha256_file(path) == item["sha256"]

PROCESSOR = BlipProcessor.from_pretrained(
    str(BLIP_SOURCE_DIR), local_files_only=True, use_fast=False
)
TOKENIZER = PROCESSOR.tokenizer
if "[DEC]" not in TOKENIZER.get_vocab():
    assert TOKENIZER.add_special_tokens({"bos_token": "[DEC]"}) == 1
else:
    TOKENIZER.bos_token = "[DEC]"
if "[ENC]" not in TOKENIZER.get_vocab():
    current = TOKENIZER.special_tokens_map.get(
        "additional_special_tokens", []
    )
    if isinstance(current, str): current = [current]
    assert TOKENIZER.add_special_tokens({
        "additional_special_tokens": list(current) + ["[ENC]"]
    }) == 1
assert TOKENIZER.convert_tokens_to_ids("[DEC]") == 30522
assert TOKENIZER.convert_tokens_to_ids("[ENC]") == 30523
assert len(TOKENIZER) == 30524


def load_model():
    model, info = BlipForConditionalGeneration.from_pretrained(
        str(BLIP_SOURCE_DIR), local_files_only=True,
        output_loading_info=True
    )
    unexpected = set(info.get("unexpected_keys", []))
    assert unexpected.issubset({
        "text_decoder.bert.embeddings.position_ids"
    })
    assert not info.get("missing_keys", [])
    assert not info.get("mismatched_keys", [])
    assert not info.get("error_msgs", [])
    return model.to(DEVICE)


class CaptionDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(row["image_path"]) as source:
            image = source.convert("RGB")
        item = PROCESSOR(
            images=image, text=row["caption"], return_tensors="pt",
            truncation=True, max_length=40
        )
        return {
            "image_id": row["image_id"],
            "annotation_id": row["annotation_id"],
            "pixel_values": item["pixel_values"][0],
            "input_ids": item["input_ids"][0],
            "attention_mask": item["attention_mask"][0],
        }


class ImageDataset(Dataset):
    def __init__(self):
        self.ids = sorted(VAL_PATHS)
    def __len__(self): return len(self.ids)
    def __getitem__(self, index):
        image_id = self.ids[index]
        with Image.open(VAL_PATHS[image_id]) as source:
            image = source.convert("RGB")
        item = PROCESSOR(images=image, return_tensors="pt")
        return {"image_id": image_id, "pixel_values": item["pixel_values"][0]}


def caption_collate(rows):
    inputs = pad_sequence(
        [x["input_ids"] for x in rows], batch_first=True,
        padding_value=TOKENIZER.pad_token_id
    )
    masks = pad_sequence(
        [x["attention_mask"] for x in rows], batch_first=True,
        padding_value=0
    )
    labels = inputs.clone()
    labels[labels == TOKENIZER.pad_token_id] = -100
    return {
        "image_ids": torch.tensor([x["image_id"] for x in rows]),
        "annotation_ids": torch.tensor([x["annotation_id"] for x in rows]),
        "pixel_values": torch.stack([x["pixel_values"] for x in rows]),
        "input_ids": inputs, "attention_mask": masks, "labels": labels,
    }


def image_collate(rows):
    return {
        "image_ids": torch.tensor([x["image_id"] for x in rows]),
        "pixel_values": torch.stack([x["pixel_values"] for x in rows]),
    }


def training_loader(epoch, cursor=0):
    return DataLoader(
        CaptionDataset(selected_epoch_records(epoch, cursor)),
        batch_size=2, shuffle=False, num_workers=2, pin_memory=True,
        collate_fn=caption_collate, persistent_workers=False
    )


VAL_CAPTION_DATASET = CaptionDataset([
    {
        "annotation_id": int(x["annotation_id"]),
        "image_id": int(x["image_id"]), "caption": x["caption"],
        "image_path": VAL_PATHS[int(x["image_id"])],
    } for x in VAL_RECORDS
])
VAL_IMAGE_DATASET = ImageDataset()
MODEL = load_model()
assert sum(x.numel() for x in MODEL.parameters()) == 247444600
train_smoke = next(iter(training_loader(0)))
val_smoke = next(iter(DataLoader(
    VAL_CAPTION_DATASET, batch_size=2, shuffle=False, num_workers=2,
    collate_fn=caption_collate
)))
assert list(train_smoke["pixel_values"].shape) == [2, 3, 384, 384]
assert list(val_smoke["pixel_values"].shape) == [2, 3, 384, 384]

write_json(AUDIT_DIR / "phase3g_cell03_model_data.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "parameters": 247444600, "tokenizer_size": len(TOKENIZER),
    "image_size": 384, "maximum_text_length": 40,
    "physical_batch": 2, "effective_batch": 32,
    "model_loaded": True, "optimizer_step": False,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72); print("PHASE 03G CELL 3 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("Model                     :", EXPECTED_REPO)
print("Parameters                : 247444600")
print("Tokenizer vocabulary      :", len(TOKENIZER))
print("Image/text                : 384 x 384 / 40")
print("Physical/effective batch  : 2 / 32")
print("Train/Validation batch    : PASSED")
print("Optimizer step            : False")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 8 cells passed")
print("Next                      : optimizer, checkpoint and pilot")


## Cell 4 — Optimizer, checkpoint/resume và discarded pilot

### Công việc

- dựng AdamW, warmup-cosine, FP16 GradScaler;
- token-balanced accumulation và overflow retry ≤10;
- định nghĩa atomic last/model-only best checkpoint;
- chạy một discarded pilot rồi reload pristine model.

### Thời gian dự kiến

1–3 phút.

### Tiến độ hiện tại

Cell 4/8 — xác minh production update engine.

### Đầu ra dự kiến

```text
Status                    : PASSED
Pilot optimizer step      : PASSED
Atomic checkpoint         : READY
Initial model restored    : True
Progress                  : 4 / 8 cells passed
```


In [ ]:
STARTED = time.perf_counter()
import torch.nn.functional as F

STEPS_PER_EPOCH = math.ceil(113287 / 32)
TOTAL_STEPS = STEPS_PER_EPOCH * 10
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)
LAST_CHECKPOINT = CHECKPOINT_DIR / "teacher_b_seed123_last.pt"
BEST_CHECKPOINT = CHECKPOINT_DIR / "teacher_b_seed123_best.pt"
STEP_LOG = LOG_DIR / "teacher_b_seed123_optimizer_steps.jsonl"
HISTORY_CSV = LOG_DIR / "teacher_b_seed123_epoch_history.csv"


def make_optimizer(model):
    vision = list(model.vision_model.parameters())
    vision_ids = {id(x) for x in vision}
    text = [x for x in model.parameters() if id(x) not in vision_ids]
    return torch.optim.AdamW([
        {"params": vision, "lr": 1e-5, "name": "vision"},
        {"params": text, "lr": 5e-5, "name": "text"},
    ], betas=(0.9, 0.999), eps=1e-8, weight_decay=0.05)


def lr_factor(step):
    if step < WARMUP_STEPS:
        return max((step + 1) / WARMUP_STEPS, 1 / WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(
        TOTAL_STEPS - WARMUP_STEPS, 1
    )
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * progress))


def make_runtime(model):
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    try:
        scaler = torch.amp.GradScaler("cuda", init_scale=1024.0)
    except (TypeError, AttributeError):
        scaler = torch.cuda.amp.GradScaler(init_scale=1024.0)
    return optimizer, scheduler, scaler


def to_device(batch):
    return {
        key: value.to(DEVICE, non_blocking=True)
        if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def update_group(model, optimizer, scheduler, scaler, cpu_batches):
    retries = 0
    while True:
        optimizer.zero_grad(set_to_none=True)
        token_sum, loss_sum = 0, 0.0
        old_scale = float(scaler.get_scale())
        for cpu_batch in cpu_batches:
            batch = to_device(cpu_batch)
            valid = int(batch["labels"][:, 1:].ne(-100).sum().item())
            with torch.autocast("cuda", dtype=torch.float16):
                output = model(
                    pixel_values=batch["pixel_values"],
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"], return_dict=True
                )
                loss = output.loss
            assert torch.isfinite(loss)
            scaler.scale(loss * valid / 1024.0).backward()
            token_sum += valid
            loss_sum += float(loss.detach().item()) * valid
            del batch, output, loss
        scaler.unscale_(optimizer)
        multiplier = 1024.0 / token_sum
        for parameter in model.parameters():
            if parameter.grad is not None:
                parameter.grad.mul_(multiplier)
        grad = float(torch.nn.utils.clip_grad_norm_(
            model.parameters(), 1.0
        ).item())
        scaler.step(optimizer)
        scaler.update()
        new_scale = float(scaler.get_scale())
        overflow = (new_scale < old_scale) or (not math.isfinite(grad))
        if overflow:
            retries += 1
            if retries > 10:
                raise RuntimeError(
                    "FP16 overflow retry limit exceeded: "
                    f"{retries - 1}/10 retries used; "
                    f"old_scale={old_scale}, new_scale={new_scale}, "
                    f"gradient_norm={grad}"
                )
            print(
                f"FP16 overflow retry {retries}/10 | "
                f"scale {old_scale:g} -> {new_scale:g}"
            )
            continue
        assert math.isfinite(grad)
        scheduler.step()
        return loss_sum / token_sum, grad, token_sum, retries


def atomic_save(payload, destination):
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    torch.save(payload, temporary); os.replace(temporary, destination)


def rng_state():
    return {
        "python": random.getstate(), "numpy": np.random.get_state(),
        "cpu": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all(),
    }


def save_last(model, optimizer, scheduler, scaler, progress, history, best):
    atomic_save({
        "schema_version": "teacher_b_seed123_checkpoint_v1",
        "trusted_project_checkpoint": True, "model": model.state_dict(),
        "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(), "rng": rng_state(),
        "progress": progress, "history": history, "best_result": best,
        "contract_sha256": sha256_file(TRAINING_CONTRACT_PATH),
        "test_set_status": "sealed", "test_set_used": False,
    }, LAST_CHECKPOINT)


def load_last(path, model, optimizer, scheduler, scaler):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    assert payload["trusted_project_checkpoint"] is True
    assert payload["test_set_used"] is False
    model.load_state_dict(payload["model"], strict=True)
    optimizer.load_state_dict(payload["optimizer"])
    scheduler.load_state_dict(payload["scheduler"])
    scaler.load_state_dict(payload["scaler"])
    random.setstate(payload["rng"]["python"])
    np.random.set_state(payload["rng"]["numpy"])
    torch.set_rng_state(payload["rng"]["cpu"])
    torch.cuda.set_rng_state_all(payload["rng"]["cuda"])
    return payload["progress"], payload["history"], payload["best_result"]


def save_best_model(model, best_result):
    atomic_save({
        "schema_version": "teacher_b_seed123_best_model_v1",
        "model": model.state_dict(), "best_result": best_result,
        "repo": EXPECTED_REPO,
        "revision": EXPECTED_REVISION, "seed": SEED,
        "test_set_status": "sealed", "test_set_used": False,
    }, BEST_CHECKPOINT)


OPTIMIZER, SCHEDULER, SCALER = make_runtime(MODEL)
pilot_iter = iter(training_loader(0))
pilot_batches = [next(pilot_iter), next(pilot_iter)]
MODEL.train()
pilot_loss, pilot_grad, pilot_tokens, pilot_retries = update_group(
    MODEL, OPTIMIZER, SCHEDULER, SCALER, pilot_batches
)
assert pilot_retries <= 10 and math.isfinite(pilot_loss)
del MODEL, OPTIMIZER, SCHEDULER, SCALER, pilot_batches, pilot_iter
gc.collect(); torch.cuda.empty_cache(); set_seeds(SEED)
MODEL = load_model()
OPTIMIZER, SCHEDULER, SCALER = make_runtime(MODEL)
assert len(OPTIMIZER.state) == 0 and SCHEDULER.last_epoch == 0

write_json(AUDIT_DIR / "phase3g_cell04_trainer_pilot.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "pilot_loss": pilot_loss, "pilot_gradient_norm": pilot_grad,
    "pilot_valid_tokens": pilot_tokens, "pilot_overflow_retries": pilot_retries,
    "optimizer_steps_per_epoch": STEPS_PER_EPOCH,
    "total_optimizer_steps": TOTAL_STEPS, "warmup_steps": WARMUP_STEPS,
    "atomic_last": True, "model_only_best": True,
    "initial_model_restored": True, "training_started": False,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72); print("PHASE 03G CELL 4 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("Pilot loss / grad         :", pilot_loss, "/", pilot_grad)
print("Pilot overflow retries    :", pilot_retries)
print("Steps/epoch / total       :", STEPS_PER_EPOCH, "/", TOTAL_STEPS)
print("Warmup steps              :", WARMUP_STEPS)
print("Atomic last/model best    : READY / READY")
print("Initial model restored    : True")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 8 cells passed")
print("Next                      : metric and Validation engine")


## Cell 5 — Official COCO metrics và production Validation

### Công việc

- cài `pycocoevalcap==1.2` từ offline wheel;
- teacher-forced CE trên 25.010 caption;
- Beam-3 trên 5.000 ảnh duy nhất;
- tính BLEU-1/4, METEOR, ROUGE-L, CIDEr-D;
- smoke-test engine, không mở Test.

### Thời gian dự kiến

1–3 phút cho smoke test; mỗi full Validation khoảng 20–45 phút.

### Tiến độ hiện tại

Cell 5/8 — khóa engine đánh giá và selection.

### Đầu ra dự kiến

```text
Status                    : PASSED
COCO metrics              : 5 / 5 WIRED
Beam repeat exact         : True
Selection                 : Validation CIDEr-D
Progress                  : 5 / 8 cells passed
```


In [ ]:
STARTED = time.perf_counter()
try:
    from pycocoevalcap.bleu.bleu import Bleu
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                    "--no-deps", str(PYCOCO_WHEEL)], check=True)
    from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer


def compute_metrics(refs, hyps):
    tokenizer = PTBTokenizer()
    refs_t, hyps_t = tokenizer.tokenize(refs), tokenizer.tokenize(hyps)
    bleu, _ = Bleu(4).compute_score(refs_t, hyps_t)
    meteor = Meteor()
    try: meteor_value, _ = meteor.compute_score(refs_t, hyps_t)
    finally:
        if hasattr(meteor, "close"): meteor.close()
    rouge, _ = Rouge().compute_score(refs_t, hyps_t)
    cider, _ = Cider().compute_score(refs_t, hyps_t)
    return {"BLEU-1": float(bleu[0]), "BLEU-4": float(bleu[3]),
            "METEOR": float(meteor_value), "ROUGE-L": float(rouge),
            "CIDEr-D": float(cider)}


def run_validation(model, epoch):
    model.eval()
    ce_sum, token_count = 0.0, 0
    caption_loader = DataLoader(
        VAL_CAPTION_DATASET, batch_size=8, shuffle=False, num_workers=2,
        pin_memory=True, collate_fn=caption_collate
    )
    with torch.inference_mode():
        for cpu in caption_loader:
            batch = to_device(cpu)
            valid = int(batch["labels"][:, 1:].ne(-100).sum().item())
            with torch.autocast("cuda", dtype=torch.float16):
                output = model(
                    pixel_values=batch["pixel_values"],
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"], return_dict=True
                )
            ce_sum += float(output.loss.item()) * valid
            token_count += valid
    hypotheses, prediction_rows = {}, []
    image_loader = DataLoader(
        VAL_IMAGE_DATASET, batch_size=8, shuffle=False, num_workers=2,
        pin_memory=True, collate_fn=image_collate
    )
    with torch.inference_mode():
        for cpu in image_loader:
            batch = to_device(cpu)
            first = model.generate(
                pixel_values=batch["pixel_values"], num_beams=3,
                do_sample=False, length_penalty=0.7, max_new_tokens=31,
                min_new_tokens=2, early_stopping=True
            )
            captions = PROCESSOR.batch_decode(
                first, skip_special_tokens=True
            )
            for image_id, token_ids, caption in zip(
                cpu["image_ids"].tolist(), first.cpu().tolist(), captions
            ):
                caption = caption.strip()
                assert caption and image_id not in TEST_IDS
                hypotheses[image_id] = [{"caption": caption}]
                prediction_rows.append({
                    "image_id": image_id, "caption": caption,
                    "token_ids": token_ids
                })
    assert len(hypotheses) == 5000
    metrics = compute_metrics(dict(REFERENCES), hypotheses)
    assert all(math.isfinite(x) for x in metrics.values())
    result = {
        "epoch": epoch, "validation_ce": ce_sum / token_count,
        **metrics, "prediction_count": len(hypotheses),
        "test_set_status": "sealed", "test_set_used": False
    }
    write_json(
        VALIDATION_DIR / f"teacher_b_seed123_epoch{epoch:02d}.json",
        {"result": result, "predictions": prediction_rows,
         "references_used": 25010, "selection_split": "validation",
         "test_set_status": "sealed", "test_set_used": False}
    )
    return result


smoke_ids = sorted(REFERENCES)[:32]
smoke_refs = {x: REFERENCES[x] for x in smoke_ids}
smoke_hyps = {x: [REFERENCES[x][0]] for x in smoke_ids}
smoke_metrics = compute_metrics(smoke_refs, smoke_hyps)
assert len(smoke_metrics) == 5
smoke_images = next(iter(DataLoader(
    VAL_IMAGE_DATASET, batch_size=2, shuffle=False, num_workers=2,
    collate_fn=image_collate
)))
MODEL.eval()
with torch.inference_mode():
    pixels = smoke_images["pixel_values"].to(DEVICE)
    first = MODEL.generate(pixel_values=pixels, num_beams=3, do_sample=False,
        length_penalty=0.7, max_new_tokens=31, min_new_tokens=2)
    second = MODEL.generate(pixel_values=pixels, num_beams=3, do_sample=False,
        length_penalty=0.7, max_new_tokens=31, min_new_tokens=2)
assert torch.equal(first, second)

write_json(AUDIT_DIR / "phase3g_cell05_validation_engine.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "backend": "pycocoevalcap 1.2", "metrics": list(smoke_metrics),
    "validation_images": 5000, "validation_references": 25010,
    "beam_size": 3, "beam_repeat_exact": True,
    "selection": "Validation CIDEr-D", "test_metrics": False,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72); print("PHASE 03G CELL 5 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("Metric backend            : pycocoevalcap 1.2")
print("Validation images/refs    : 5000 / 25010")
print("COCO metrics              : 5 / 5 WIRED")
print("Beam repeat exact         : True")
print("Selection                 : Validation CIDEr-D")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 8 cells passed")
print("Next                      : launch/resume full training")


## Cell 6 — Full training hoặc exact resume

### Công việc

- tự động resume từ local/external `last` checkpoint;
- train tối đa 10 epoch, effective batch 32;
- lưu atomic last mỗi 500 optimizer step;
- full Validation cuối epoch, lưu model-only best;
- early stop từ epoch 3 với patience 3.

### Thời gian dự kiến

Nhiều phiên Kaggle; khoảng 5–10 giờ/epoch trên T4. Save Version và gắn output
cũ để resume nếu hết thời gian.

### Tiến độ hiện tại

Cell 6/8 — chạy huấn luyện dài hạn.

### Đầu ra dự kiến

```text
Status                    : TRAINING COMPLETE
Best epoch                : <1..10>
Best Validation CIDEr-D   : <finite>
Last/best checkpoint      : SAVED
Progress                  : 6 / 8 cells passed
```


In [ ]:
STARTED = time.perf_counter()
from itertools import islice

progress = {"epoch": 0, "cursor": 0, "optimizer_step": 0,
            "overflow_retries": 0, "no_improve": 0}
history, best_result = [], None
resume_path = LAST_CHECKPOINT if LAST_CHECKPOINT.is_file() else (
    RESUME_CANDIDATES[0] if RESUME_CANDIDATES else None
)
run_mode = "fresh"
if resume_path is not None:
    run_mode = "local_resume" if resume_path == LAST_CHECKPOINT else "external_resume"
    if run_mode == "external_resume":
        old_root = resume_path.parents[1]
        for relative in [
            "logs/teacher_b_seed123_optimizer_steps.jsonl",
            "logs/teacher_b_seed123_epoch_history.csv",
        ]:
            source, target = old_root / relative, OUTPUT_ROOT / relative
            if source.is_file() and not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, target)
        old_validation = old_root / "validation"
        if old_validation.is_dir():
            for source in old_validation.glob("*.json"):
                target = VALIDATION_DIR / source.name
                if not target.exists(): shutil.copy2(source, target)
    progress, history, best_result = load_last(
        resume_path, MODEL, OPTIMIZER, SCHEDULER, SCALER
    )
    if STEP_LOG.is_file():
        retained = []
        for line in STEP_LOG.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            if int(row["optimizer_step"]) <= progress["optimizer_step"]:
                retained.append(row)
        deduplicated = {
            int(row["optimizer_step"]): row for row in retained
        }
        assert sorted(deduplicated) == list(
            range(1, progress["optimizer_step"] + 1)
        )
        STEP_LOG.write_text(
            "".join(
                json.dumps(deduplicated[step], sort_keys=True) + "\n"
                for step in sorted(deduplicated)
            ),
            encoding="utf-8",
        )


def better(candidate, current):
    if current is None: return True
    if candidate["CIDEr-D"] > current["CIDEr-D"] + 0.001: return True
    if abs(candidate["CIDEr-D"] - current["CIDEr-D"]) <= 0.001:
        if candidate["BLEU-4"] > current["BLEU-4"]: return True
        if candidate["BLEU-4"] == current["BLEU-4"]:
            if candidate["validation_ce"] < current["validation_ce"]: return True
    return False


def append_step(row):
    with open(STEP_LOG, "a", encoding="utf-8") as target:
        target.write(json.dumps(row, sort_keys=True) + "\n")


def write_history(rows):
    fields = ["epoch", "train_ce", "validation_ce", "BLEU-1", "BLEU-4",
              "METEOR", "ROUGE-L", "CIDEr-D", "optimizer_step",
              "overflow_retries", "training_seconds"]
    with open(HISTORY_CSV, "w", newline="", encoding="utf-8") as target:
        writer = csv.DictWriter(target, fieldnames=fields)
        writer.writeheader(); writer.writerows(rows)


MAX_EPOCHS, ACCUMULATION, CHECKPOINT_INTERVAL = 10, 16, 500
training_started_at = time.perf_counter()
while progress["epoch"] < MAX_EPOCHS:
    epoch = progress["epoch"]
    MODEL.train()
    loader = iter(training_loader(epoch, progress["cursor"]))
    epoch_loss_sum, epoch_tokens = 0.0, 0
    epoch_started = time.perf_counter()
    while True:
        group = list(islice(loader, ACCUMULATION))
        if not group: break
        mean_loss, grad, tokens, retries = update_group(
            MODEL, OPTIMIZER, SCHEDULER, SCALER, group
        )
        images = sum(len(batch["image_ids"]) for batch in group)
        progress["cursor"] += images
        progress["optimizer_step"] += 1
        progress["overflow_retries"] += retries
        epoch_loss_sum += mean_loss * tokens
        epoch_tokens += tokens
        row = {
            "epoch": epoch + 1, "cursor": progress["cursor"],
            "optimizer_step": progress["optimizer_step"],
            "mean_token_ce": mean_loss, "valid_tokens": tokens,
            "gradient_norm": grad, "overflow_retries": retries,
            "lr_vision": OPTIMIZER.param_groups[0]["lr"],
            "lr_text": OPTIMIZER.param_groups[1]["lr"],
            "test_set_used": False,
        }
        append_step(row)
        if progress["optimizer_step"] % 25 == 0:
            print(
                f"epoch={epoch+1:02d} step={progress['optimizer_step']:05d} "
                f"cursor={progress['cursor']:06d} loss={mean_loss:.5f}"
            )
        if (progress["optimizer_step"] % CHECKPOINT_INTERVAL == 0 or
                progress["cursor"] >= 113287):
            save_last(MODEL, OPTIMIZER, SCHEDULER, SCALER,
                      progress, history, best_result)
    assert progress["cursor"] == 113287
    validation = run_validation(MODEL, epoch + 1)
    epoch_row = {
        "epoch": epoch + 1, "train_ce": epoch_loss_sum / epoch_tokens,
        **{k: validation[k] for k in [
            "validation_ce", "BLEU-1", "BLEU-4", "METEOR",
            "ROUGE-L", "CIDEr-D"
        ]},
        "optimizer_step": progress["optimizer_step"],
        "overflow_retries": progress["overflow_retries"],
        "training_seconds": time.perf_counter() - epoch_started,
    }
    history.append(epoch_row)
    if better(validation, best_result):
        best_result = dict(validation)
        progress["no_improve"] = 0
        save_best_model(MODEL, best_result)
    else:
        progress["no_improve"] += 1
    progress["epoch"] += 1
    progress["cursor"] = 0
    write_history(history)
    save_last(MODEL, OPTIMIZER, SCHEDULER, SCALER,
              progress, history, best_result)
    print(
        f"EPOCH {epoch+1:02d} | train={epoch_row['train_ce']:.5f} | "
        f"val={validation['validation_ce']:.5f} | "
        f"CIDEr-D={validation['CIDEr-D']:.6f} | "
        f"best={best_result['CIDEr-D']:.6f}"
    )
    if progress["epoch"] >= 3 and progress["no_improve"] >= 3:
        break

assert history and best_result is not None
assert LAST_CHECKPOINT.is_file() and BEST_CHECKPOINT.is_file()
TRAINING_FINISHED = True
write_json(AUDIT_DIR / "phase3g_cell06_full_training.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "run_mode": run_mode, "completed_epochs": progress["epoch"],
    "optimizer_steps": progress["optimizer_step"],
    "overflow_retries": progress["overflow_retries"],
    "best_result": best_result, "history_rows": len(history),
    "last_checkpoint": str(LAST_CHECKPOINT),
    "best_checkpoint": str(BEST_CHECKPOINT),
    "full_training_started": True, "full_training_finished": True,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72); print("PHASE 03G CELL 6 STATUS"); print("=" * 72)
print("Status                    : TRAINING COMPLETE")
print("Run mode                  :", run_mode)
print("Completed epochs          :", progress["epoch"])
print("Optimizer steps           :", progress["optimizer_step"])
print("Overflow retries          :", progress["overflow_retries"])
print("Best epoch                :", best_result["epoch"])
print("Best Validation CIDEr-D   :", best_result["CIDEr-D"])
print("Last/best checkpoint      : SAVED / SAVED")
print("Test Set status           : SEALED")
print("Progress                  : 6 / 8 cells passed")
print("Next                      : quality gate and run report")


## Cell 7 — Quality gate và seed-42 run report

### Công việc

- đối chiếu best checkpoint với epoch history;
- yêu cầu CIDEr-D ≥ 0,998355;
- kiểm tra overflow retry ≤10 và metric hữu hạn;
- phát hành kết quả cho phép confirmation seeds.

### Thời gian dự kiến

1–5 phút.

### Tiến độ hiện tại

Cell 7/8 — quyết định Teacher B có đạt quality gate hay không.

### Đầu ra dự kiến

```text
Status                    : PASSED
Quality gate              : PASSED
Qualified for KD          : True
Confirmation seeds        : AUTHORIZED
Progress                  : 7 / 8 cells passed
```


In [ ]:
STARTED = time.perf_counter()
assert TRAINING_FINISHED is True
assert best_result["epoch"] in [row["epoch"] for row in history]
best_history = next(
    row for row in history if row["epoch"] == best_result["epoch"]
)
for key in ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D"]:
    assert math.isfinite(best_result[key])
    assert best_history[key] == best_result[key]
QUALITY_THRESHOLD = 0.9833811369118861
quality_margin = best_result["CIDEr-D"] - QUALITY_THRESHOLD
QUALITY_GATE_PASSED = quality_margin >= 0
assert QUALITY_GATE_PASSED, {
    "CIDEr-D": best_result["CIDEr-D"],
    "required": QUALITY_THRESHOLD,
}
assert progress["overflow_retries"] <= 10

RUN_RESULT_PATH = OUTPUT_ROOT / "teacher_b_seed123_run_result.json"
write_json(RUN_RESULT_PATH, {
    "status": "passed", "created_at_utc": now_utc(),
    "teacher_id": "B", "seed": 123, "model": EXPECTED_REPO,
    "revision": EXPECTED_REVISION, "completed_epochs": progress["epoch"],
    "optimizer_steps": progress["optimizer_step"],
    "overflow_retries": progress["overflow_retries"],
    "best_result": best_result, "quality_threshold": QUALITY_THRESHOLD,
    "quality_margin": quality_margin, "quality_gate_passed": True,
    "qualified_for_kd": True,
    "direct_logit_kd": "prohibited",
    "feature_kd": "allowed_after_projection_contract",
    "sequence_kd": "allowed",
    "provenance_label": "external COCO-pretrained teacher",
    "selection_split": "validation", "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT_DIR / "phase3g_cell07_quality_gate.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "best_epoch": best_result["epoch"],
    "best_validation_cider_d": best_result["CIDEr-D"],
    "quality_threshold": QUALITY_THRESHOLD, "quality_margin": quality_margin,
    "quality_gate_passed": True, "qualified_for_kd": True,
    "confirmation_seeds_authorized": [123, 2026],
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72); print("PHASE 03G CELL 7 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("Best epoch                :", best_result["epoch"])
print("Best Validation CIDEr-D   :", best_result["CIDEr-D"])
print("Quality threshold         :", QUALITY_THRESHOLD)
print("Quality margin            :", quality_margin)
print("Quality gate              : PASSED")
print("Qualified for KD          : True")
print("Confirmation seeds        : AUTHORIZED")
print("Test Set status           : SEALED")
print("Progress                  : 7 / 8 cells passed")
print("Next                      : package full run")


## Cell 8 — Package full run và COMPLETE marker

### Công việc

- hash last/best checkpoints, logs, Validation và audit;
- tạo manifest CSV/JSON/SHA256SUMS;
- tạo seed-42 COMPLETE marker và confirmation authorization;
- xác minh output cuối, không mở Test.

### Thời gian dự kiến

3–12 phút do checkpoint có dung lượng nhiều GB.

### Tiến độ hiện tại

Cell 8/8 — đóng gói Phase 03G.

### Đầu ra bắt buộc

```text
Status                    : COMPLETE
Teacher B seed 123         : COMPLETE
Quality gate              : PASSED
Manifest failures         : 0
Test Set status           : SEALED
Progress                  : 8 / 8 cells passed
```


In [ ]:
STARTED = time.perf_counter()
assert QUALITY_GATE_PASSED and RUN_RESULT_PATH.is_file()
COMPLETE_MARKER = OUTPUT_ROOT / "TEACHER_B_SEED123_FULL_RUN_COMPLETE.json"
CONFIRMATION_MARKER = (
    OUTPUT_ROOT / "TEACHER_B_SEED123_PARALLEL_RUN_RECORDED.json"
)
assert not COMPLETE_MARKER.exists() and not CONFIRMATION_MARKER.exists()

artifact_paths = sorted([
    path for parent in [
        AUDIT_DIR, CONFIG_DIR, CHECKPOINT_DIR, LOG_DIR, VALIDATION_DIR
    ] for path in parent.rglob("*") if path.is_file()
] + [RUN_RESULT_PATH])
manifest_rows = [{
    "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
    "size_bytes": path.stat().st_size, "sha256": sha256_file(path)
} for path in artifact_paths]

MANIFEST_CSV = MANIFEST_DIR / "phase3g_manifest.csv"
with open(MANIFEST_CSV, "w", newline="", encoding="utf-8") as target:
    writer = csv.DictWriter(
        target, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader(); writer.writerows(manifest_rows)
MANIFEST_JSON = MANIFEST_DIR / "phase3g_manifest.json"
write_json(MANIFEST_JSON, {
    "schema_version": "phase3g_manifest_v1",
    "created_at_utc": now_utc(), "artifact_count": len(manifest_rows),
    "artifacts": manifest_rows
})
SHA256SUMS = MANIFEST_DIR / "phase3g_SHA256SUMS.txt"
SHA256SUMS.write_text("".join(
    f"{x['sha256']}  {x['relative_path']}\n" for x in manifest_rows
), encoding="utf-8")
failures = []
for row in manifest_rows:
    path = OUTPUT_ROOT / row["relative_path"]
    if (not path.is_file() or path.stat().st_size != row["size_bytes"]
            or sha256_file(path) != row["sha256"]):
        failures.append(row["relative_path"])
assert not failures
manifest_refs = {
    "csv": {"relative_path": MANIFEST_CSV.relative_to(OUTPUT_ROOT).as_posix(),
            "sha256": sha256_file(MANIFEST_CSV)},
    "json": {"relative_path": MANIFEST_JSON.relative_to(OUTPUT_ROOT).as_posix(),
             "sha256": sha256_file(MANIFEST_JSON)},
    "sha256sums": {
        "relative_path": SHA256SUMS.relative_to(OUTPUT_ROOT).as_posix(),
        "sha256": sha256_file(SHA256SUMS)
    },
}
write_json(COMPLETE_MARKER, {
    "marker_version": "teacher_b_seed123_full_run_complete_v1",
    "created_at_utc": now_utc(), "status": "complete",
    "teacher_id": "B", "seed": 123, "model": EXPECTED_REPO,
    "revision": EXPECTED_REVISION,
    "completed_epochs": progress["epoch"],
    "optimizer_steps": progress["optimizer_step"],
    "overflow_retries": progress["overflow_retries"],
    "best_result": best_result, "quality_threshold": QUALITY_THRESHOLD,
    "quality_margin": quality_margin, "quality_gate_passed": True,
    "qualified_for_kd": True, "selection_split": "validation",
    "best_checkpoint": {
        "relative_path": BEST_CHECKPOINT.relative_to(OUTPUT_ROOT).as_posix(),
        "sha256": sha256_file(BEST_CHECKPOINT)
    },
    "last_checkpoint": {
        "relative_path": LAST_CHECKPOINT.relative_to(OUTPUT_ROOT).as_posix(),
        "sha256": sha256_file(LAST_CHECKPOINT)
    },
    "manifests": manifest_refs,
    "provenance_label": "external COCO-pretrained teacher",
    "test_set_status": "sealed", "test_set_used": False
})
write_json(CONFIRMATION_MARKER, {
    "marker_version": "teacher_b_confirmation_authorized_v1",
    "created_at_utc": now_utc(), "status": "ready",
    "teacher_id": "B", "seeds": [123],
    "seed42_complete": {
        "relative_path": COMPLETE_MARKER.relative_to(OUTPUT_ROOT).as_posix(),
        "sha256": sha256_file(COMPLETE_MARKER)
    },
    "authorized_next_action": (
        "Accept this run only after seed 42 independently passes; "
        "training contract; keep Held-out Test sealed."
    ),
    "test_set_status": "sealed", "test_set_used": False
})
print("=" * 72); print("PHASE 03G CELL 8 STATUS"); print("=" * 72)
print("Status                    : COMPLETE")
print("Teacher B seed 123         : COMPLETE")
print("Best epoch / CIDEr-D      :", best_result["epoch"], "/", best_result["CIDEr-D"])
print("Quality gate              : PASSED")
print("Confirmation seeds        : 123 / 2026 AUTHORIZED")
print("Manifest rows             :", len(manifest_rows))
print("Manifest failures         :", len(failures))
print("Best checkpoint SHA-256   :", sha256_file(BEST_CHECKPOINT))
print("Last checkpoint SHA-256   :", sha256_file(LAST_CHECKPOINT))
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Progress                  : 8 / 8 cells passed")
